In [25]:
import pandas as pd
import numpy as np

df = pd.read_csv('Recruitment_master.csv')
print(f"Loaded: {df.shape}")
print(df.columns.tolist())

Loaded: (500, 49)
['candidate_id', 'first_name', 'last_name', 'nationality', 'job_function', 'seniority_level', 'years_of_experience', 'expected_salary_gbp', 'assessment_score', 'key_skills', 'english_proficiency', 'remote_preference', 'registration_date', 'location', 'education_level', 'availability_notice_days', 'visa_status', 'source_channel', 'profile_status', 'placement_id', 'placement_status', 'time_to_hire_days', 'offered_salary_gbp', 'offer_accepted', 'client_satisfaction_score', 'rehire_likelihood', 'placement_fee_gbp', 'estimated_fee_value_gbp', 'interview_rounds', 'hiring_manager', 'recruiter_assigned', 'client_company', 'client_industry', 'role_title', 'contract_type', 'placement_region', 'start_date', 'end_date', 'probation_outcome', 'performance_rating', 'invoice_status', 'replacement_required', 'notes', 'salary_gap', 'salary_match_ratio', 'skills_count', 'days_since_registration', 'interview_efficiency', 'was_placed']


In [26]:
# ==== EDA revealed skills_count, interview_efficiency, and performance_rating all showed NaN in correlations ====
# to fix them 

# skills_count — re-derive safely
df['skills_count'] = df['key_skills'].str.split(',').str.len().fillna(0).astype(int)

# interview_efficiency — avoid divide-by-zero from Phase 6 zero-fills
df['interview_efficiency'] = np.where(
    df['interview_rounds'] > 0,
    df['time_to_hire_days'] / df['interview_rounds'],
    np.nan
)

# performance_rating — check what it looks like
print(df['performance_rating'].value_counts(dropna=False).head(10))
print(f"\nskills_count nulls: {df['skills_count'].isna().sum()}")
print(f"interview_efficiency nulls: {df['interview_efficiency'].isna().sum()}")

performance_rating
NaN     206
7.0      50
8.0      48
9.0      41
6.0      31
10.0     25
5.0      25
2.0      23
4.0      23
3.0      17
Name: count, dtype: int64

skills_count nulls: 0
interview_efficiency nulls: 130


diagnosed and repaired three columns that showed NaN correlation in the EDA:

skills_count — re-derived by splitting key_skills on commas and counting the pieces. Result: 0 nulls — fixed perfectly.

interview_efficiency — the Phase 6 zero-fill had turned the 130 unplaced candidates' interview_rounds to 0, causing division by zero. The fix used np.where() to only calculate the ratio when interview_rounds > 0, and assign NaN otherwise. Result: 130 nulls — this is correct and expected (those are the unplaced candidates who had no interviews).

performance_rating — the print revealed:

206 nulls — more than the 130 unplaced candidates. This means some placed candidates also have no performance rating yet (perhaps still in probation or contract end date hasn't passed). This needs careful handling — do not fill with median blindly as it would distort the ML signal.

In [27]:
# ==== Drop leaky features ====

# These columns contain post-placement information — they wouldn't exist at prediction time, 
# so using them in ML would give artificially perfect results


leaky_cols = [
    'salary_match_ratio',     # 0.991 correlation — derived from offered_salary which only exists post-placement
    'salary_gap',             # 0.894 — same issue
    'offered_salary_gbp',     # 0.793 — only known after placement
    'time_to_hire_days',      # 0.791 — only known after placement
    'interview_rounds',       # 0.825 — only known after placement
    'client_satisfaction_score',  # 0.809 — post-placement outcome
    'placement_status',       # direct label leak
    'placement_id',           # meaningless for ML
    'placement_fee_gbp',      # post-placement
    'estimated_fee_value_gbp' # post-placement
]

# Only drop columns that actually exist
leaky_cols = [c for c in leaky_cols if c in df.columns]
df_ml = df.drop(columns=leaky_cols)
print(f"Shape after dropping leaky cols: {df_ml.shape}")
print(f"Dropped: {leaky_cols}")

Shape after dropping leaky cols: (500, 39)
Dropped: ['salary_match_ratio', 'salary_gap', 'offered_salary_gbp', 'time_to_hire_days', 'interview_rounds', 'client_satisfaction_score', 'placement_status', 'placement_id', 'placement_fee_gbp', 'estimated_fee_value_gbp']


In [28]:
# === Encode Categoricals === numbers needed and not strings

from sklearn.preprocessing import LabelEncoder

# --- Ordinal encoding (order matters) ---
seniority_order = {'intern': 0, 'entry level': 1, 'junior': 2, 'mid': 3, 'senior': 4}
df_ml['seniority_encoded'] = df_ml['seniority_level'].map(seniority_order)

# --- One-hot encoding (no order) ---
ohe_cols = ['remote_preference', 'rehire_likelihood', 'offer_accepted']
ohe_cols = [c for c in ohe_cols if c in df_ml.columns]

df_ml = pd.get_dummies(df_ml, columns=ohe_cols, drop_first=True)
print(f"Shape after encoding: {df_ml.shape}")
print(df_ml.dtypes.value_counts())

Shape after encoding: (500, 44)
object     25
int64      10
bool        7
float64     2
Name: count, dtype: int64


Two things happened here:

Ordinal encoding — seniority_level was mapped to numbers {intern:0, entry level:1, junior:2, mid:3, senior:4} preserving the career progression order in a single new column seniority_encoded.

One-hot encoding — remote_preference, rehire_likelihood, and offer_accepted were exploded into binary columns using pd.get_dummies(). The drop_first=True removes one category per column to avoid multicollinearity (the "dummy variable trap"). This is why the shape went from 39 → 44 columns (new binary columns added).

Problem spotted: After encoding, there are still 25 object columns remaining. The OHE only encoded 3 columns — many more categoricals survived.

In [29]:
# === Drop Non-Numeric Columns for ML Use ====

# Keeping the raw strings for reference in EDA, but creating a clean numeric-only ML dataframe:

# Columns to drop for ML (text/ID/date — not useful as raw values)
drop_for_ml = ['candidate_id', 'full_name', 'email', 'key_skills',
               'registration_date', 'placement_start_date',
               'seniority_level', 'notes']

drop_for_ml = [c for c in drop_for_ml if c in df_ml.columns]
df_ml = df_ml.drop(columns=drop_for_ml)

# Convert any remaining object columns
for col in df_ml.select_dtypes(include='object').columns:
    print(f"  Still object: {col} — {df_ml[col].unique()[:5]}")

print(f"\nFinal ML-ready shape: {df_ml.shape}")
print(f"Target balance:\n{df_ml['was_placed'].value_counts()}")

  Still object: first_name — ['Leo' 'Reuben' 'Fatima' 'Theo' 'Eleanor']
  Still object: last_name — ['Walker' "O'Connor" 'Nakamura' 'Zhang' 'Singh']
  Still object: nationality — ['Turkish' 'Moroccan' 'Dutch' 'Nepalese' 'South African']
  Still object: job_function — ['Software Engineer' 'Frontend Developer' 'Full Stack Developer'
 'Machine Learning Engineer' 'Technical Support Engineer']
  Still object: location — ['Birmingham' 'Derry' 'Dublin' 'Newcastle' 'Glasgow']
  Still object: education_level — ["Bachelor's" 'Diploma' "Master's" 'Professional Certification' 'PhD']
  Still object: visa_status — ['EU settled status' 'Student visa' 'British or Irish citizen'
 'Requires sponsorship' 'Skilled Worker visa']
  Still object: source_channel — ['LinkedIn' 'Recruiter Outreach' 'University Partner' 'Referral'
 'Company Website']
  Still object: profile_status — ['placed' 'inactive' 'under review' 'active']
  Still object: hiring_manager — ['Emma Collins' nan 'Hannah Foster' 'Michael Turner'

The cell dropped candidate_id, key_skills, registration_date, seniority_level, and notes. However, the output exposed a major problem — there are still 25 object-type columns that need handling before ML:

| Column                                               | Issue                                      | Action needed                                                                                        |
| ---------------------------------------------------- | ------------------------------------------ | ---------------------------------------------------------------------------------------------------- |
| first_name, last_name                                | Personally identifiable, no ML value       | Drop                                                                                                 |
| nationality, location                                | Categorical — many unique values           | One-hot encode or drop                                                                               |
| job_function, education_level                        | Categorical with ML value                  | One-hot encode                                                                                       |
| visa_status, source_channel                          | Categorical with ML value                  | One-hot encode                                                                                       |
| profile_status                                       | Categorical                                | One-hot encode                                                                                       |
| remote_preference, offer_accepted, rehire_likelihood | Already OHE'd but still showing as object  | They weren't actually encoded — the OHE ran on df before 
these columns were treated as 
category type |
| hiring_manager, recruiter_assigned                   | High cardinality names — 10+ unique values | Drop or label encode                                                                                 |
| client_company, role_title                           | High cardinality                           | Drop for now                                                                                         |
| contract_type, placement_region                      | Useful categoricals                        | One-hot encode                                                                                       |
| start_date, end_date                                 | Dates stored as string                     | Drop (post-placement data)                                                                           |
| probation_outcome, invoice_status                    | Post-placement data                        | Drop                                                                                                 |
| replacement_required                                 | Post-placement                             | Drop                                                                                                 |

In [30]:
# === FIX: Smart Null Handling by Column Type ===

# Step 1: Columns to DROP entirely (post-placement or too high cardinality)
drop_cols = [
    'first_name', 'last_name',               # PII, no ML value
    'hiring_manager', 'recruiter_assigned',  # high cardinality names
    'client_company', 'role_title',          # high cardinality, post-placement
    'start_date', 'end_date',                # post-placement dates
    'probation_outcome', 'invoice_status',   # post-placement outcomes
    'replacement_required'                   # post-placement
]
drop_cols = [c for c in drop_cols if c in df_ml.columns]
df_ml = df_ml.drop(columns=drop_cols)
print(f"Shape after dropping high-cardinality/post-placement cols: {df_ml.shape}")

# Step 2: One-hot encode remaining object columns with ML value
ohe_remaining = ['nationality', 'job_function', 'location', 'education_level',
                 'visa_status', 'source_channel', 'profile_status',
                 'contract_type', 'placement_region',
                 'remote_preference', 'offer_accepted', 'rehire_likelihood']
ohe_remaining = [c for c in ohe_remaining if c in df_ml.columns]

df_ml = pd.get_dummies(df_ml, columns=ohe_remaining, drop_first=True)
print(f"Shape after encoding remaining categoricals: {df_ml.shape}")

# Step 3: Fill performance_rating — use median only for PLACED candidates
# (NaN for unplaced is structurally correct — keep them as 0)
df_ml['performance_rating'] = df_ml.apply(
    lambda row: row['performance_rating'] if pd.notna(row['performance_rating'])
    else (df_ml.loc[df_ml['was_placed']==1, 'performance_rating'].median() 
          if row['was_placed']==1 else 0),
    axis=1
)

# Step 4: Fill interview_efficiency for unplaced candidates with 0
df_ml['interview_efficiency'] = df_ml['interview_efficiency'].fillna(0)

# Step 5: Final null check
nulls = df_ml.isnull().sum()
nulls = nulls[nulls > 0]
print(f"\n{'✅ No nulls — ML ready!' if len(nulls)==0 else '⚠️ Remaining nulls:'}")
if len(nulls) > 0:
    print(nulls)

print(f"\nFinal shape: {df_ml.shape}")
print(f"Object cols remaining: {list(df_ml.select_dtypes('object').columns)}")

Shape after dropping high-cardinality/post-placement cols: (500, 28)
Shape after encoding remaining categoricals: (500, 123)

⚠️ Remaining nulls:
client_industry    130
dtype: int64

Final shape: (500, 123)
Object cols remaining: ['client_industry']


| Problem                           | Old approach             | Fixed approach                                     |
| --------------------------------- | ------------------------ | -------------------------------------------------- |
| String columns (names, companies) | Tried median() → crashed | Drop them entirely                                 |
| performance_rating 206 nulls      | Fill all with median     | Fill placed candidates with median; unplaced get 0 |
| interview_efficiency 130 nulls    | Fill all with median     | Fill unplaced with 0 (they had no interviews)      |
| Remaining 25 object columns       | Not handled              | OHE all useful categoricals, drop rest             |



In [31]:
# client_industry: fill NaN with 'Unknown' then one-hot encode
df_ml['client_industry'] = df_ml['client_industry'].fillna('Unknown')
df_ml = pd.get_dummies(df_ml, columns=['client_industry'], drop_first=True)

# Final verification
nulls = df_ml.isnull().sum()
nulls = nulls[nulls > 0]
obj_cols = list(df_ml.select_dtypes('object').columns)

print(f"{'✅ No nulls — ML ready!' if len(nulls)==0 else '⚠️ Still nulls:'}")
if len(nulls) > 0: print(nulls)
print(f"Object cols remaining: {obj_cols if obj_cols else 'None ✅'}")
print(f"Final shape: {df_ml.shape}")
print(f"\nTarget balance:\n{df_ml['was_placed'].value_counts()}")

✅ No nulls — ML ready!
Object cols remaining: None ✅
Final shape: (500, 134)

Target balance:
was_placed
1    370
0    130
Name: count, dtype: int64


In [32]:
# === REMOVE REMAINING LEAKY FEATURES ===

# These only exist after placement has already happened
additional_leaky = [
    'interview_efficiency',   # derived from time_to_hire / interview_rounds
    'performance_rating',     # post-placement review score
    'offer_accepted_yes',     # decision made after offer — post-placement process
    'offer_accepted_no',      # same
    'offer_accepted_pending', # same
    'rehire_likelihood_yes',  # post-placement assessment
    'rehire_likelihood_no',
    'rehire_likelihood_maybe',
    'client_industry_Unknown', # this encodes "not placed" — direct label leak
]

additional_leaky = [c for c in additional_leaky if c in df_ml.columns]
df_ml = df_ml.drop(columns=additional_leaky)
print(f"Shape after removing remaining leaky features: {df_ml.shape}")

# Re-save
df_ml.to_csv('Recruitment_ml_ready.csv', index=False)
print(f"✅ Re-saved Recruitment_ml_ready.csv — shape: {df_ml.shape}")

Shape after removing remaining leaky features: (500, 125)
✅ Re-saved Recruitment_ml_ready.csv — shape: (500, 125)


In [33]:
# Check which original file contract_type came from
print("In candidates:", 'contract_type' in pd.read_csv('candidate_profiles.csv').columns)
print("In placements:", 'contract_type' in pd.read_csv('placement_outcomes.csv').columns)

In candidates: False
In placements: True


In [34]:
# Drop contract_type — confirmed post-placement (from placement_outcomes.csv)
contract_cols = [c for c in df_ml.columns if 'contract_type' in c]
print(f"Dropping: {contract_cols}")
df_ml = df_ml.drop(columns=contract_cols)


Dropping: ['contract_type_fixed-term', 'contract_type_permanent', 'contract_type_temp-to-perm']


In [35]:
# === Drop placement_region — post-placement geographic data ===
region_cols = [c for c in df_ml.columns if 'placement_region' in c]
print(f"Dropping: {region_cols}")
df_ml = df_ml.drop(columns=region_cols)


print("In candidates:", 'client_industry' in pd.read_csv('candidate_profiles.csv').columns)
print("In placements:", 'client_industry' in pd.read_csv('placement_outcomes.csv').columns)



Dropping: ['placement_region_Northern Ireland', 'placement_region_Remote', 'placement_region_Republic of Ireland', 'placement_region_Scotland', 'placement_region_Wales']
In candidates: False
In placements: True


In [36]:
# === Drop client_industry — confirmed post-placement (from placement_outcomes.csv) ===
industry_cols = [c for c in df_ml.columns if 'client_industry' in c]
print(f"Dropping {len(industry_cols)} client_industry columns: {industry_cols}")

df_ml = df_ml.drop(columns=industry_cols)
print(f"Shape after drop: {df_ml.shape}")

# Re-save final clean version
df_ml.to_csv('Recruitment_ml_ready.csv', index=False)
print(f"✅ Final re-saved — shape: {df_ml.shape}")
print(f"Remaining object cols: {list(df_ml.select_dtypes('object').columns)}")
print(f"Null count: {df_ml.isnull().sum().sum()}")

Dropping 11 client_industry columns: ['client_industry_Cybersecurity', 'client_industry_Education', 'client_industry_Energy', 'client_industry_Financial Services', 'client_industry_Healthcare', 'client_industry_Logistics', 'client_industry_Manufacturing', 'client_industry_Public Sector', 'client_industry_Retail', 'client_industry_Technology', 'client_industry_Telecommunications']
Shape after drop: (500, 106)
✅ Final re-saved — shape: (500, 106)
Remaining object cols: []
Null count: 0
